In [1]:
import os

In [2]:
%pwd

'/Users/harshpatel/Desktop/Projects/End-to-End-Kidney-Disease-Classification-Deep-Learning-Project/notebooks'

In [3]:
os.chdir('../')

In [4]:
%pwd

'/Users/harshpatel/Desktop/Projects/End-to-End-Kidney-Disease-Classification-Deep-Learning-Project'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from KidneyDiseaseClassification.constants import *
from KidneyDiseaseClassification.utils.common import read_yaml,create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config

In [ ]:
import os
import zipfile
import gdown
import sys
from KidneyDiseaseClassification.utils.logger import logger
from KidneyDiseaseClassification.utils.exception import CustomException
from KidneyDiseaseClassification.utils.common import get_size

In [ ]:
class DataIngestion:
    def __init__(self, config :DataIngestionConfig):
        self.config = config
    
    def download_file(self)-> str:
        '''
        Fetch data from the URL
        '''

        try:
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs("artifacts/data_ingestion" ,exist_ok=True)
            logger.info(f"Downloading data from {dataset_url} into file {zip_download_dir}")
            
            file_id = dataset_url.split("/")[-2]
            prefix = "https://drive.google.com/uc?export=download&id="
            gdown.download(url=prefix+file_id,output=zip_download_dir)

            logger.info(f"Downloaded data from {dataset_url} into file {zip_download_dir}")
        except Exception as e:
            raise CustomException(e,sys)
        
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path,exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file,'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [10]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise CustomException(e,sys)

[2026-06-02 15:06:49,980] INFO common: yaml file: config/config.yaml loaded successfully
[2026-06-02 15:06:49,981] INFO common: yaml file: params.yaml loaded successfully
[2026-06-02 15:06:49,981] INFO common: created directory at: artifacts
[2026-06-02 15:06:49,982] INFO common: created directory at: artifacts/data_ingestion


Downloading...
From (original): https://drive.google.com/uc?id=18a4g866_eRzXtNOCYW9Mc61Dm6SWHuVB
From (redirected): https://drive.google.com/uc?id=18a4g866_eRzXtNOCYW9Mc61Dm6SWHuVB&confirm=t&uuid=436eae46-e730-41f9-8a69-475fc34df524
To: /Users/harshpatel/Desktop/Projects/End-to-End-Kidney-Disease-Classification-Deep-Learning-Project/artifacts/data_ingestion/data.zip
100%|██████████| 944M/944M [01:22<00:00, 11.4MB/s] 
